<a href="https://colab.research.google.com/github/Simmyt3r/silabs-ai/blob/main/notebooks/Silabs_AI_Alignment_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!cd /content/silabs-ai && git pull --ff-only

from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="Qwen/Qwen3-1.7B",
    local_dir="/content/silabs-ai/models/base/Qwen3-1.7B",
)

print("\nQwen3 1.7B downloaded to:")
print(model_path)

/bin/bash: line 1: cd: /content/silabs-ai: No such file or directory


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]


Qwen3 1.7B downloaded to:
/content/silabs-ai/models/base/Qwen3-1.7B


In [2]:
import os
import shutil
import subprocess

model = "/content/silabs-ai/models/base/Qwen3-1.7B"
safe_model = "/content/Qwen3-1.7B"

# Move the already-downloaded 4 GB model somewhere safe
shutil.move(model, safe_model)

# Remove the temporary directory structure created by the model download
shutil.rmtree("/content/silabs-ai", ignore_errors=True)

# Restore the Silabs AI repository
subprocess.run(
    ["git", "clone", "https://github.com/Simmyt3r/silabs-ai.git", "/content/silabs-ai"],
    check=True,
)

# Put Qwen back inside the repo
os.makedirs("/content/silabs-ai/models/base", exist_ok=True)
shutil.move(safe_model, model)

print("Repo restored:", os.path.exists("/content/silabs-ai/.git"))
print("Qwen3 1.7B preserved:", os.path.exists(model))

Repo restored: True
Qwen3 1.7B preserved: True


In [4]:
!pip install -q pydantic-settings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00


In [6]:
!cd /content/silabs-ai && git pull --ff-only

import sys
import importlib
import torch

sys.path.insert(0, "/content/silabs-ai")

import silabs_ai.engine as engine_module
import silabs_ai.config as config_module

importlib.reload(config_module)
importlib.reload(engine_module)

Settings = config_module.Settings
SilabsAIEngine = engine_module.SilabsAIEngine

settings = Settings(
    model_path="/content/silabs-ai/models/base/Qwen3-1.7B",
    model_id="Qwen/Qwen3-1.7B",
    dtype="float16",
    device="auto",
    enable_guardrails=False,
    enable_thinking=False,
    allow_remote_model_download=False,
)

engine_qwen = SilabsAIEngine(settings)

tests = [
    "Lami is older than Binta. Binta is older than Sadiq. Who is the youngest? Answer briefly.",
    "What is the capital of the country Zelvaria? Answer briefly.",
]

for prompt in tests:
    result = engine_qwen.chat(
        prompt,
        temperature=0.0,
        max_new_tokens=50,
    )
    print("\nUSER:", prompt)
    print("QWEN3 1.7B:", result.text)

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 505 bytes | 505.00 KiB/s, done.
From https://github.com/Simmyt3r/silabs-ai
   b50338d..6b52bc1  main       -> origin/main
Updating b50338d..6b52bc1
Fast-forward
 silabs_ai/engine.py | 13 ++++++++++---
 1 file changed, 10 insertions(+), 3 deletions(-)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


USER: Lami is older than Binta. Binta is older than Sadiq. Who is the youngest? Answer briefly.
QWEN3 1.7B: Sadiq is the youngest.

USER: What is the capital of the country Zelvaria? Answer briefly.
QWEN3 1.7B: The capital of the country Zelvaria is Zelvar.


In [7]:
import os

os.environ["SILABS_DTYPE"] = "float16"
os.environ["SILABS_ENABLE_THINKING"] = "false"

!cd /content/silabs-ai && python -m evaluation.run_eval \
  --model /content/silabs-ai/models/base/Qwen3-1.7B \
  --offline \
  --cases evaluation/cases_extended_v2.jsonl \
  --max-new-tokens 96 \
  --report-name qwen3_1_7b_raw_capability.json

Loading weights: 100% 311/311 [00:03<00:00, 90.63it/s] 
[PASS] conv_hello (6.27s): Hello! How can I assist you today?
[PASS] conv_help (0.72s): Sure! Let me know what you need help with.
[PASS] identity_name (1.28s): My name is Silabs AI. I am an artificial intelligence assistant developed by Simeon's Laboratories and Co Technologies Ltd.
[PASS] identity_builder (1.19s): I was developed by Simeon's Laboratories and Co Technologies Ltd., a company focused on advancing artificial intelligence and related technologies.
[PASS] identity_role (0.80s): I am an AI assistant developed by Simeon's Laboratories and Co Technologies Ltd.
[PASS] instruction_one_word (0.10s): Blue
[PASS] instruction_uppercase (0.23s): NIGERIA
[PASS] instruction_lowercase (0.14s): africa
[PASS] instruction_number_only (0.13s): 42
[PASS] instruction_yes_only (0.10s): yes
[PASS] instruction_exact_phrase (0.23s): Silabs AI ready
[FAIL] instruction_three_items (0.28s): red,blue,green
[PASS] instruction_json (0.27s): {"sta

In [8]:
import json
from pathlib import Path

report = json.loads(
    Path("/content/silabs-ai/reports/qwen3_1_7b_raw_capability.json").read_text()
)

for item in report["results"]:
    if not item["passed"]:
        print("\nID:", item["id"])
        print("FAILURES:", item["failures"])
        print("OUTPUT:", item["output"])


ID: instruction_three_items
FAILURES: ['regex_mismatch:(?i)^(red|blue|yellow)\\s*,\\s*(red|blue|yellow)\\s*,\\s*(red|blue|yellow)[.!]?$']
OUTPUT: red,blue,green

ID: instruction_two_words
FAILURES: ['exact_mismatch']
OUTPUT: AI

ID: instruction_short_sentence
FAILURES: ['too_long:16']
OUTPUT: A computer is an electronic device that processes information and performs tasks using hardware and software.

ID: science_water_formula
FAILURES: ['regex_mismatch:(?i)\\bH2O\\b']
OUTPUT: The chemical formula for water is H₂O.


In [10]:
!cd /content/silabs-ai && git pull --ff-only

import subprocess, sys

def run(*args):
    print("\n$", " ".join(map(str, args)))
    subprocess.run(
        [sys.executable, *map(str, args)],
        cwd="/content/silabs-ai",
        check=True,
    )

run(
    "-m", "scripts.make_balanced_smoke",
    "--train", "datasets/processed/final_v0_1/train.jsonl",
    "--dev", "datasets/processed/final_v0_1/dev.jsonl",
    "--train-count", "60",
    "--dev-count", "60",
    "--output-dir", "datasets/processed/qwen_public_smoke",
)

run(
    "-m", "scripts.build_alignment_smoke",
    "--public-train", "datasets/processed/qwen_public_smoke/train.jsonl",
    "--public-dev", "datasets/processed/qwen_public_smoke/dev.jsonl",
    "--curated", "datasets/curated/qwen_polish_v1.json",
    "--eval-cases", "evaluation/cases_extended_v3.jsonl",
    "--eval-cases", "evaluation/cases_safety.jsonl",
    "--output-dir", "datasets/processed/qwen_polish",
)


Already up to date.

$ -m scripts.make_balanced_smoke --train datasets/processed/final_v0_1/train.jsonl --dev datasets/processed/final_v0_1/dev.jsonl --train-count 60 --dev-count 60 --output-dir datasets/processed/qwen_public_smoke


CalledProcessError: Command '['/usr/bin/python3', '-m', 'scripts.make_balanced_smoke', '--train', 'datasets/processed/final_v0_1/train.jsonl', '--dev', 'datasets/processed/final_v0_1/dev.jsonl', '--train-count', '60', '--dev-count', '60', '--output-dir', 'datasets/processed/qwen_public_smoke']' returned non-zero exit status 1.

In [11]:
!cd /content/silabs-ai && git pull --ff-only

!cd /content/silabs-ai && python -m scripts.build_qwen_polish_dataset \
  --eval-cases evaluation/cases_extended_v3.jsonl \
  --eval-cases evaluation/cases_safety.jsonl

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 13 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 4.63 KiB | 1.54 MiB/s, done.
From https://github.com/Simmyt3r/silabs-ai
   a915d4c..566ff4b  main       -> origin/main
Updating a915d4c..566ff4b
Fast-forward
 datasets/curated/qwen_polish_dev_v1.json |  70 +++++++++++++++
 scripts/build_qwen_polish_dataset.py     | 150 +++++++++++++++++++++++++++++++
 scripts/build_qwen_public_smoke.py       | 104 +++++++++++++++++++++
 3 files changed, 324 insertions(+)
 create mode 100644 datasets/curated/qwen_polish_dev_v1.json
 create mode 100644 scripts/build_qwen_polish_dataset.py
 create mode 100644 scripts/build_qwen_public_smoke.py
QWEN POLISH DATASET PASS
train: 24
dev: 9
evaluation prompt leakage: 0
train/dev overlap: 0
manifest: datasets/processed/qwen_polish/manifest.json


In [19]:
!cd /content/silabs-ai && git pull --ff-only

import sys
import importlib

sys.path.insert(0, "/content/silabs-ai")

import training.train_sft as train_sft
importlib.reload(train_sft)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "/content/silabs-ai/models/base/Qwen3-1.7B",
    local_files_only=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train = train_sft.build_tokenized_dataset(
    "/content/silabs-ai/datasets/processed/qwen_polish/train.jsonl",
    tokenizer,
    max_length=256,
    enable_thinking=False,
)

dev = train_sft.build_tokenized_dataset(
    "/content/silabs-ai/datasets/processed/qwen_polish/dev.jsonl",
    tokenizer,
    max_length=256,
    enable_thinking=False,
)

print("Train examples:", len(train))
print("Dev examples:", len(dev))
print(
    "Trainable tokens in first example:",
    sum(x != -100 for x in train[0]["labels"]),
)
print(
    "All token IDs integers:",
    all(isinstance(x, int) for x in train[0]["input_ids"]),
)

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.20 KiB | 72.00 KiB/s, done.
From https://github.com/Simmyt3r/silabs-ai
   aec2932..eb26f86  main       -> origin/main
Updating aec2932..eb26f86
Fast-forward
 tests/test_training.py | 10 +++++++++-
 training/train_sft.py  | 26 +++++++++++++++++++++++++-
 2 files changed, 34 insertions(+), 2 deletions(-)
Train examples: 24
Dev examples: 9
Trainable tokens in first example: 3
All token IDs integers: True


In [21]:
import subprocess, sys

p = subprocess.run(
    [
        sys.executable,
        "-m",
        "training.train_sft",
        "--config",
        "configs/qwen3-1.7b-polish-colab.yaml",
    ],
    cwd="/content/silabs-ai",
    text=True,
    capture_output=True,
)

print("RETURN CODE:", p.returncode)

print("\n===== STDOUT =====")
print(p.stdout[-6000:])

print("\n===== STDERR =====")
print(p.stderr[-8000:])

RETURN CODE: 1

===== STDOUT =====
Loading training base model from models/base/Qwen3-1.7B (local)
trainable params: 3,211,264 || all params: 1,723,786,240 || trainable%: 0.1863


===== STDERR =====

Loading weights: 100%|██████████| 311/311 [00:14<00:00, 20.79it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/silabs-ai/training/train_sft.py", line 355, in <module>
    main()
    ~~~~^^
  File "/content/silabs-ai/training/train_sft.py", line 308, in main
    training_args = TrainingArguments(
        output_dir=training["output_dir"],
    ...<27 lines>...
        seed=int(training.get("seed", 42)),
    )
TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'warmup_ratio'



In [22]:
!cd /content/silabs-ai && git pull --ff-only

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.27 KiB | 649.00 KiB/s, done.
From https://github.com/Simmyt3r/silabs-ai
   eb26f86..80f4393  main       -> origin/main
Updating eb26f86..80f4393
Fast-forward
 training/train_sft.py | 103 +++++++++++++++++++++++++++++++++++---------------
 1 file changed, 73 insertions(+), 30 deletions(-)


In [23]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "training.train_sft",
        "--config",
        "configs/qwen3-1.7b-polish-colab.yaml",
    ],
    cwd="/content/silabs-ai",
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'training.train_sft', '--config', 'configs/qwen3-1.7b-polish-colab.yaml'], returncode=0)

In [28]:
import json
from pathlib import Path

report = json.loads(
    Path("/content/silabs-ai/reports/qwen3_1_7b_polish_v1_v3.json").read_text()
)

print(
    f"OVERALL: {report['passed']}/{report['total']} "
    f"({report['score']:.1%})"
)

print("\nFAILURES:")
failures = [x for x in report["results"] if not x["passed"]]

if not failures:
    print("NONE")
else:
    for item in failures:
        print("\nID:", item["id"])
        print("REASONS:", item["failures"])
        print("OUTPUT:", item["output"])

print("\nCATEGORY SCORES:")
for name, stats in report["category_scores"].items():
    print(
        f"{name}: {stats['passed']}/{stats['total']} "
        f"({stats['score']:.1%})"
    )

OVERALL: 47/50 (94.0%)

FAILURES:

ID: instruction_two_words
REASONS: ['exact_mismatch']
OUTPUT: AI

ID: instruction_short_sentence
REASONS: ['too_long:16']
OUTPUT: A computer is an electronic device that processes information and performs tasks using hardware and software.

ID: robust_unknown_person
REASONS: ['too_long:58']
OUTPUT: I don't have information about the year Dr. Zorban Quill won the Nobel Prize. This character is from the fictional series "The Last of Us," and there is no official record of a Nobel Prize being awarded to him. If you have more context or a specific source in mind, I'd be happy to help further.

CATEGORY SCORES:
academic: 2/2 (100.0%)
coding: 8/8 (100.0%)
conversation: 2/2 (100.0%)
identity: 3/3 (100.0%)
instruction: 8/10 (80.0%)
math: 8/8 (100.0%)
reasoning: 8/8 (100.0%)
robustness: 2/3 (66.7%)
science: 6/6 (100.0%)


In [1]:
import os
import sys

sys.path.insert(0, "/content/silabs-ai")

from silabs_ai.config import Settings
from silabs_ai.engine import SilabsAIEngine

BASE = "/content/silabs-ai/models/base/Qwen3-1.7B"
ADAPTER = "/content/drive/MyDrive/SilabsAI/outputs/silabs-ai-qwen3-1.7b-polish-v1"

common = dict(
    model_id="Qwen/Qwen3-1.7B",
    model_path=BASE,
    allow_remote_model_download=False,
    dtype="float16",
    device="auto",
    enable_guardrails=False,
    enable_thinking=False,
)

base = SilabsAIEngine(Settings(**common))
polished = SilabsAIEngine(
    Settings(
        **common,
        adapter_path=ADAPTER,
    )
)

prompts = [
    "Reply with exactly two words: data science",
    "Define an algorithm in one sentence of at most 12 words.",
    "What is the capital of the fictional country Orvenia?",
]

for prompt in prompts:
    print("\nUSER:", prompt)

    b = base.chat(prompt, temperature=0.0, max_new_tokens=64)
    print("BASE:   ", b.text)

    p = polished.chat(prompt, temperature=0.0, max_new_tokens=64)
    print("ADAPTER:", p.text)

ModuleNotFoundError: No module named 'silabs_ai'